In [1]:
import pandas as pd
import numpy as np

In [2]:
clean_data = pd.read_parquet("../cleaned_data/clean_yellow_tripdata_2026-01.parquet")

In [3]:
df = clean_data.copy()

In [4]:
df.sample(4)

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_type_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,...,extra_amount,mta_tax_amount,tip_amount,tolls_amount,improvement_surcharge_amount,total_amount,congestion_surcharge_amount,airport_fee_amount,cbd_congestion_fee_amount,vendor_name
2470418,2,2026-01-30 10:41:54,2026-01-30 10:56:09,1.0,1.39,1.0,N,237,75,1,...,0.0,0.5,3.50,0.0,1.0,21.00,2.5,0.0,0.00,"Curb Mobility, LLC"
2213310,2,2026-01-27 14:49:50,2026-01-27 15:02:12,2.0,1.15,1.0,N,237,239,2,...,0.0,0.5,0.00,0.0,1.0,16.10,2.5,0.0,0.00,"Curb Mobility, LLC"
1455571,2,2026-01-17 18:11:07,2026-01-17 18:27:18,1.0,5.83,1.0,N,116,241,1,...,0.0,0.5,3.00,0.0,1.0,29.20,0.0,0.0,0.00,"Curb Mobility, LLC"
1778717,2,2026-01-21 15:36:10,2026-01-21 15:52:22,1.0,1.20,1.0,N,230,142,1,...,0.0,0.5,9.82,0.0,1.0,29.47,2.5,0.0,0.75,"Curb Mobility, LLC"


# check pickup time < dropoff time

In [5]:
df = df[(df['pickup_datetime'] < df['dropoff_datetime'])]

In [6]:
df.shape

(3679819, 21)

# Check dates, not included in 2026 jan. 2025-12-31 and 2026-02-01 is not midnight trips

In [7]:
print(df[
    (df['pickup_datetime'] >= '2025-12-31') |
    (df['dropoff_datetime'] <= '2026-02-01')
].shape)
df = df[
    (df['pickup_datetime'] >= '2025-12-31') |
    (df['dropoff_datetime'] <= '2026-02-01')
]


(3679819, 21)


# calculate trip duration

In [8]:
df['trip_duration'] = df['dropoff_datetime'] - df['pickup_datetime']

# remove trips having trip duration less than 2 min

In [9]:
df = df[(df['trip_duration'] >= pd.Timedelta(minutes=2))]

In [10]:
df.shape

(3618604, 22)

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 3618604 entries, 0 to 3724888
Data columns (total 22 columns):
 #   Column                        Dtype          
---  ------                        -----          
 0   vendor_id                     int8           
 1   pickup_datetime               datetime64[us] 
 2   dropoff_datetime              datetime64[us] 
 3   passenger_count               float64        
 4   trip_distance                 float64        
 5   fare_type_id                  float64        
 6   store_and_fwd_flag            str            
 7   pickup_location_id            int32          
 8   dropoff_location_id           int32          
 9   payment_type                  int64          
 10  fare_amount                   float64        
 11  extra_amount                  float64        
 12  mta_tax_amount                float64        
 13  tip_amount                    float64        
 14  tolls_amount                  float64        
 15  improvement_surcharge_amount  f

# save back to the cleaned file 

In [12]:
df.to_parquet('../cleaned_data/clean_yellow_tripdata_2026-01.parquet', index=False)

In [13]:
df.shape

(3618604, 22)

# vendor id 7 is get completely removed - means it has only dirty data

In [14]:
df['vendor_id'].value_counts()

vendor_id
2    2915371
1     699230
6       4003
Name: count, dtype: int64

# function to clean pickup and dropoff datetime

In [15]:
def clean_pickup_dropoff_datetime(df):

    df = df[(df['pickup_datetime'] < df['dropoff_datetime'])]

    df = df[
        (df['pickup_datetime'] >= '2025-12-31') |
        (df['dropoff_datetime'] <= '2026-02-01')
    ]

    df['trip_duration'] = df['dropoff_datetime'] - df['pickup_datetime']

    df = df[(df['trip_duration'] >= pd.Timedelta(minutes=2))]